# Nhóm 10 – Dự đoán tái nhập viện ở bệnh nhân đái tháo đường (Diabetes 130-US)
**Notebook tổng hợp của nhóm** (bản chính thức nằm trên nhánh `main`). Notebook chạy **script của từng thành viên, lấy trực tiếp từ nhánh của người đó**, theo thứ tự từ trên xuống. Các bước nối với nhau bằng file trong `output/`: output của bước trước là input của bước sau.

**Cách chạy:** `Runtime → Run all`. Nếu một bước lỗi, notebook **dừng ngay ở bước đó** và in lỗi, các bước sau không chạy.

| Bước | Task | Người | Script (nhánh) | Đọc | Ghi (`output/`) |
|---|---|---|---|---|---|
| 01 | Bỏ thuộc tính định danh | Dũng | `src/01_explore_and_drop_ids.py` (Dung) | `data/diabetic_data.csv` | `step1.csv`, `step1_patient_nbr.csv` |
| 02 | Thống kê khuyết, bỏ cột khuyết > 50% | Sơn | `src/02_missing_stats_and_filter.py` (Son) | `step1.csv` | `step2.csv`, `missing_rate.csv` |
| 03 | Bỏ dòng khuyết < 2%, tạo nhãn, chia 80/20 | Chiến | `src/03_data_preprocessing_splitting.py` (Chien) | `step2.csv`, `step1_patient_nbr.csv` | `train.csv`, `test.csv` |
| 04 | One-Hot (fit trên train) | Dũng | `src/04_one_hot_encoding.py` (Dung) | `train.csv`, `test.csv` | `X_train_onehot.npz`, `X_test_onehot.npz`, `onehot_columns.txt`, `y_train.csv`, `y_test.csv` |
| 05 | PCA 60 thành phần | Sơn | `src/05_pca.py` (Son) | `X_*_onehot.npz`, `onehot_columns.txt` | `X_train_pca.npy`, `X_test_pca.npy`, `pca_eigenvalues.npy`, `pca_components.npy`, `pca_feature_names.txt` |
| 06 | Chuẩn hóa sau PCA, scree plot, top 5 vector riêng | Chiến | `src/06_feature_scaling_pca_analysis.py` (Chien) | `X_*_pca.npy`, `pca_*` | `X_train_pca_scaled.npy`, `X_test_pca_scaled.npy`, `eigenvalues.png` |

**Quy ước**
- Mọi script chạy từ thư mục gốc dự án, đọc/ghi đúng tên file trong bảng. Không đổi tên file khi chưa báo cả nhóm.
- Các bước học từ dữ liệu (One-Hot, PCA, chuẩn hóa) chỉ fit trên train. `random_state = 42`.
- Muốn sửa một bước: sửa script trên nhánh của mình rồi push, chạy lại notebook là cập nhật. Muốn thêm bước mới: thêm một dòng vào bảng trên và một ô `run(...)` bên dưới.
- Mỗi bước có ghi **kết quả mong đợi**. Nếu số in ra khác, nghĩa là có bước trước đó đã thay đổi.

## 0. Cài đặt
Clone cả ba nhánh vào `/content/branches/`. Thư mục làm việc là `branches/Dung` (có dữ liệu gốc); script của Sơn và Chiến được gọi qua `../Son/src/...`, `../Chien/src/...` nên cả ba cùng đọc/ghi một thư mục `output/`.

In [ ]:
import os
REPO = "https://github.com/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project.git"
BRANCHES = ["Dung", "Son", "Chien"]

%cd /content
# Ra khỏi branches/ trước khi xóa, nếu không chạy lại ô này sẽ xóa đúng thư mục đang đứng
!rm -rf /content/branches
for b in BRANCHES:
    !git clone -q --depth 1 -b {b} {REPO} /content/branches/{b}
    !git -C /content/branches/{b} log -1 --format="{b}: %h %an %ad %s" --date=format:"%d/%m %H:%M"
assert all(os.path.isdir(f"/content/branches/{b}/src") for b in BRANCHES), "Clone lỗi, xem thông báo ở trên"

%cd /content/branches/Dung
!pip -q install -r requirements.txt -r ../Son/requirements.txt -r ../Chien/requirements.txt

In [ ]:
import subprocess, sys

def run(script):
    """Chạy một script từ thư mục gốc dự án; dừng notebook nếu script lỗi."""
    print(f"$ python {script}\n")
    r = subprocess.run([sys.executable, script], capture_output=True, text=True,
                       env={**os.environ, "MPLBACKEND": "Agg"})
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} lỗi (mã {r.returncode}), xem thông báo ở trên")

## Giai đoạn 1: Khảo sát và xử lý dữ liệu khuyết

### Bước 01: Bỏ thuộc tính định danh *(Dũng)*
**Kết quả mong đợi:** kích thước gốc `(101766, 50)` → sau khi bỏ định danh `(101766, 47)`.

In [ ]:
run("src/01_explore_and_drop_ids.py")

### Bước 02: Thống kê khuyết, bỏ cột khuyết > 50% *(Sơn)*
**Kết quả mong đợi:** bỏ `weight`, `max_glu_serum`, `A1Cresult` → `(101766, 44)`.

In [ ]:
run("../Son/src/02_missing_stats_and_filter.py")

### Bước 03: Bỏ dòng khuyết < 2%, tạo nhãn, chia train/test 80/20 *(Chiến)*
**Kết quả mong đợi:** xóa 1522 dòng → train 80195, test 20049; tỉ lệ nhãn hai tập: NO 0,537, >30 0,351, <30 0,112.

In [ ]:
run("../Chien/src/03_data_preprocessing_splitting.py")

## Giai đoạn 2: One-Hot, PCA, chuẩn hóa

### Bước 04: One-Hot Encoding *(Dũng)*
**Kết quả mong đợi:** 35 biến phân loại, 8 biến số → X_train `(80195, 2386)`, X_test `(20049, 2386)`.

In [ ]:
run("src/04_one_hot_encoding.py")

### Bước 05: PCA 60 thành phần trên ma trận tương quan *(Sơn)*
**Kết quả mong đợi:** còn 2384 chiều sau khi bỏ cột phương sai 0; 60 thành phần giải thích ≈ 0,0549 phương sai.

In [ ]:
run("../Son/src/05_pca.py")

### Bước 06: Chuẩn hóa sau PCA, biểu đồ Eigenvalues, top 5 vector riêng *(Chiến)*
**Kết quả mong đợi:** độ lệch chuẩn sau chuẩn hóa = 1; ghi `eigenvalues.png`.

In [ ]:
run("../Chien/src/06_feature_scaling_pca_analysis.py")

### Kết quả Giai đoạn 2

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir("output")):
    print(f"{os.path.getsize(f'output/{f}') / 1e6:8.2f} MB  {f}")
display(Image("output/eigenvalues.png"))

## Giai đoạn 3: Huấn luyện mô hình — chưa làm

## Giai đoạn 4: Đánh giá tổng hợp — chưa làm